# Generate Video Audio

Adds spoken audio to a **silent video** from text you provide, in any supported language.
Uses Google Cloud **Text-to-Speech** API — no GPU or model downloads needed.

**Usage:** Set `API_KEY`, `VIDEO_PATH`, `TEXT`, and `LANGUAGE_CODE` / `VOICE_NAME`, then run all cells.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q ffmpeg-python

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
API_KEY = ""  # ← your Google Cloud API key
VIDEO_PATH = "/content/drive/MyDrive/videos/input.mp4"  # silent input video

# The text to speak over the video (in the target language)
TEXT = """
Your narration text goes here. Write it in the language you want spoken.
""".strip()

# Language & voice (https://cloud.google.com/text-to-speech/docs/voices)
LANGUAGE_CODE = "en-US"       # BCP-47 language code
VOICE_NAME = "en-US-Neural2-D"  # specific voice (set to "" for auto-select)

# Common language + voice combinations:
# fmt: off
VOICE_EXAMPLES = {
    "en-US":  ["en-US-Neural2-D (M)",  "en-US-Neural2-C (F)"],
    "hi-IN":  ["hi-IN-Neural2-D (M)",  "hi-IN-Neural2-A (F)"],
    "es-ES":  ["es-ES-Neural2-B (M)",  "es-ES-Neural2-A (F)"],
    "fr-FR":  ["fr-FR-Neural2-B (M)",  "fr-FR-Neural2-A (F)"],
    "de-DE":  ["de-DE-Neural2-B (M)",  "de-DE-Neural2-A (F)"],
    "it-IT":  ["it-IT-Neural2-C (M)",  "it-IT-Neural2-A (F)"],
    "pt-BR":  ["pt-BR-Neural2-B (M)",  "pt-BR-Neural2-A (F)"],
    "ja-JP":  ["ja-JP-Neural2-C (M)",  "ja-JP-Neural2-B (F)"],
    "ko-KR":  ["ko-KR-Neural2-C (M)",  "ko-KR-Neural2-A (F)"],
    "zh-CN":  ["zh-CN-Neural2-B (M)",  "zh-CN-Neural2-A (F)"],
    "ar-XA":  ["ar-XA-Neural2-A (M)",  "ar-XA-Neural2-C (F)"],
    "ru-RU":  ["ru-RU-Neural2-B (M)",  "ru-RU-Neural2-A (F)"],
    "tr-TR":  ["tr-TR-Neural2-B (M)",  "tr-TR-Neural2-A (F)"],
    "nl-NL":  ["nl-NL-Neural2-B (M)",  "nl-NL-Neural2-A (F)"],
    "pl-PL":  ["pl-PL-Neural2-B (M)",  "pl-PL-Neural2-A (F)"],
    "vi-VN":  ["vi-VN-Neural2-D (M)",  "vi-VN-Neural2-A (F)"],
    "id-ID":  ["id-ID-Neural2-B (M)",  "id-ID-Neural2-A (F)"],
    "bn-IN":  ["bn-IN-Neural2-B (M)",  "bn-IN-Neural2-A (F)"],
    "ta-IN":  ["ta-IN-Neural2-B (M)",  "ta-IN-Neural2-A (F)"],
    "te-IN":  ["te-IN-Neural2-B (M)",  "te-IN-Neural2-A (F)"],
}
# fmt: on

assert API_KEY, "Set your Google Cloud API key above"
assert TEXT, "Provide the text to narrate"
print(f"Language: {LANGUAGE_CODE}")
print(f"Voice: {VOICE_NAME or '(auto)'}")
print(f"Text length: {len(TEXT)} chars")

In [ ]:
import os, base64, tempfile, wave, requests, ffmpeg
from pathlib import Path
from tqdm.auto import tqdm

TTS_URL = "https://texttospeech.googleapis.com/v1/text:synthesize"
TTS_RATE = 24_000


def _api(url, body):
    """POST to a Google Cloud REST endpoint and return JSON."""
    r = requests.post(url, params={"key": API_KEY}, json=body)
    if not r.ok:
        msg = r.json().get("error", {}).get("message", r.text)
        raise RuntimeError(f"API error {r.status_code}: {msg}")
    return r.json()


def _wav_duration(path):
    with wave.open(path) as w:
        return w.getnframes() / w.getframerate()


def synthesize(text, language_code, voice_name=""):
    """Synthesize speech from text. Returns path to a WAV file."""
    max_chars = 4500  # TTS API limit is 5000 bytes per request
    chunks = [text[i : i + max_chars] for i in range(0, len(text), max_chars)]
    wav_path = tempfile.mktemp(suffix=".wav")

    voice_cfg = {"languageCode": language_code}
    if voice_name:
        voice_cfg["name"] = voice_name

    if len(chunks) == 1:
        body = {
            "input": {"text": chunks[0]},
            "voice": voice_cfg,
            "audioConfig": {"audioEncoding": "LINEAR16", "sampleRateHertz": TTS_RATE},
        }
        resp = _api(TTS_URL, body)
        with open(wav_path, "wb") as f:
            f.write(base64.b64decode(resp["audioContent"]))
    else:
        with wave.open(wav_path, "wb") as out:
            for i, chunk in enumerate(tqdm(chunks, desc="Synthesizing", unit="chunk")):
                body = {
                    "input": {"text": chunk},
                    "voice": voice_cfg,
                    "audioConfig": {"audioEncoding": "LINEAR16", "sampleRateHertz": TTS_RATE},
                }
                resp = _api(TTS_URL, body)
                ptmp = tempfile.mktemp(suffix=".wav")
                with open(ptmp, "wb") as f:
                    f.write(base64.b64decode(resp["audioContent"]))
                with wave.open(ptmp) as w:
                    if i == 0:
                        out.setparams(w.getparams())
                    out.writeframes(w.readframes(w.getnframes()))
                os.remove(ptmp)

    return wav_path


def add_audio_to_video(video_path, audio_wav, output_path):
    """Mux video (keeping original visual) with generated audio, adjusting tempo to match video duration."""
    video_dur = float(ffmpeg.probe(video_path)["format"]["duration"])
    audio_dur = _wav_duration(audio_wav)

    print(f"  Video duration: {video_dur:.1f}s")
    print(f"  Audio duration: {audio_dur:.1f}s")

    vid = ffmpeg.input(video_path)
    aud = ffmpeg.input(audio_wav)

    # Adjust audio speed to match video length if within a reasonable range
    tempo = audio_dur / video_dur
    if 0.5 <= tempo <= 2.0 and abs(tempo - 1.0) > 0.05:
        print(f"  Adjusting audio tempo by {tempo:.2f}x to match video length")
        aud_stream = aud.audio.filter("atempo", tempo)
    else:
        if tempo > 2.0 or tempo < 0.5:
            print(f"  Warning: audio/video duration ratio ({tempo:.2f}x) is too extreme to adjust. Audio may not match video length.")
        aud_stream = aud.audio

    ffmpeg.output(
        vid.video, aud_stream, output_path,
        vcodec="copy", acodec="aac", strict="experimental",
    ).overwrite_output().run(quiet=True)


print("Functions ready.")

In [ ]:
# ── Generate ────────────────────────────────────────────────────────────────
video = Path(VIDEO_PATH)
assert video.exists(), f"Video not found: {VIDEO_PATH}"

lang_short = LANGUAGE_CODE.split("-")[0]
output_path = str(video.with_stem(f"{video.stem}_{lang_short}_narrated"))

print(f"Input:  {VIDEO_PATH}")
print(f"Output: {output_path}\n")

progress = tqdm(total=2, desc="Progress", unit="step")

# Step 1: Synthesize speech from text
progress.set_postfix_str("synthesizing speech")
tts_wav = synthesize(TEXT, LANGUAGE_CODE, VOICE_NAME)
print(f"Synthesized audio: {_wav_duration(tts_wav):.1f}s")
progress.update(1)

# Step 2: Add audio to video
progress.set_postfix_str("muxing audio with video")
add_audio_to_video(VIDEO_PATH, tts_wav, output_path)
os.remove(tts_wav)
progress.update(1)

progress.set_postfix_str("done")
progress.close()
print(f"\nVideo with narration saved to:\n{output_path}")